In [ ]:
import sys
import os
# Add the parent directory to the Python path
sys.path.append(os.path.abspath('..'))
from simu_PSF_polarMFM_JAX import *
from simu_PSF_polarMFM import *
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import torch
device = torch.device('cuda')
print(jax.devices())

In [ ]:
N_photons=jax.device_put(1000, device=gpu) # number of photons collected 
N=80 # discretization of the BFP
l_pixel=16 #pixel size in micrometer
NA=1.4 # numerical aperture
mag=100 # first magnification
lambd=617 # wavelength
f_tube=200 # tube lens focal
MAG=200/150 # second magnification

In [ ]:
x, y, th1, phi, (Ex0, Ex1, Ex2), (Ey0, Ey1, Ey2), r, r_cut, k, f_o = vectorial_BFP_perfect_focus_jax(N, NA, mag, lambd, f_tube)

In [ ]:
zernike_base = generate_zernike_base_jax(r_cut, N, zernike_order=4, device=device)

In [ ]:
u, v, Npadding = padding_jax(r, r_cut, k, f_o)
print(Npadding)

In [ ]:
xp = jnp.array([0.3,0.,0.,0.,0.])
yp = jnp.array([0.,0.,0.,0.,0.])
zp = jnp.array([3.*0.617,1.,1.,1.,1.])
d = -3.6*0.617/0.8+0.35
second_plane = jnp.array([0.3, 0, -0.3])
polar_projections = jnp.array([0., 45., 0.])

In [ ]:
th1 = pad_jax(th1, Npadding)
phi = pad_jax(phi, Npadding)
Ex0 = pad_jax(Ex0, Npadding)
Ex1 = pad_jax(Ex1, Npadding)
Ex2 = pad_jax(Ex2, Npadding)
Ey0 = pad_jax(Ey0, Npadding)
Ey1 = pad_jax(Ey1, Npadding)
Ey2 = pad_jax(Ey2, Npadding)
zernike_base = pad_jax(zernike_base, Npadding)

In [ ]:
zernike_base.shape

In [ ]:
Mj = compute_M_jax(xp, yp, zp, d, x, y, th1, phi, Ex0, Ex1, Ex2, Ey0, Ey1, Ey2, u, v, zernike_base, jnp.zeros((3,15)), jnp.zeros((3,15)),  second_plane=second_plane
                  , polar_projections=polar_projections)

In [ ]:
rho = jnp.array([45.,30.,40.,80.,100.])
eta = jnp.array([45.,60.,30.,80.,50.])
delta = jnp.array([50.,100.,100.,100.,100.])
N_photons = jnp.array([1000.,1000.,1000.,1000.,1000.])

In [ ]:
psf_jax = PSF_jax(rho, eta, delta, Mj, N_photons)

In [ ]:
plt.imshow(psf_jax[0,1,0])
plt.xlim((70,85))
plt.ylim((70,85))

In [ ]:
psf_jax.shape

In [ ]:
Mj.shape

In [ ]:
plt.rcParams['figure.figsize'] = [10,7]
fig, ax = plt.subplots(3,2)
ax[0,0].imshow(jnp.real(Mj[0,0,0,0,0]+Mj[0,0,0,0,0]))
ax[0,0].set_xlim((64,95))
ax[0,0].set_ylim((62,92))
ax[0,1].imshow(jnp.real(Mj[0,0,0,0,1]+Mj[0,0,0,1,0]))
ax[0,1].set_xlim((64,95))
ax[0,1].set_ylim((62,92))
ax[1,0].imshow(jnp.real(Mj[0,0,0,1,1]+Mj[0,0,0,1,1]))
ax[1,0].set_xlim((64,95))
ax[1,0].set_ylim((62,92))
ax[1,1].imshow(jnp.real(Mj[0,0,0,1,2]+Mj[0,0,0,2,1]))
ax[1,1].set_xlim((64,95))
ax[1,1].set_ylim((62,92))
ax[2,0].imshow(jnp.real(Mj[0,0,0,2,2]+Mj[0,0,0,2,2]))
ax[2,0].set_xlim((64,95))
ax[2,0].set_ylim((62,92))
ax[2,1].imshow(jnp.real(Mj[0,0,0,0,2]+Mj[0,0,0,2,0]))
ax[2,1].set_xlim((64,95))
ax[2,1].set_ylim((62,92))

In [ ]:
key = jax.random.PRNGKey(0)

In [ ]:
psf_noise_jax = noise_jax(key, psf_jax, QE=0.95, EM=200, b=5.0, sigma_b=0.1, sigma_r=1.0, bias=5.0)

In [ ]:
plt.imshow(psf_noise_jax[0,1,0])
plt.xlim((70,85))
plt.ylim((70,85))